# Extract the List of Post Codes in Alberta from Wikipedia

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd

## Get HTML of the Postal Codes List

In [2]:
url = 'https://en.wikipedia.org/wiki/List_of_postal_codes_of_Canada:_T'

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/142.0.0.0 Safari/537.36"
}

page = requests.get(url, headers = headers)

In [3]:
soup = BeautifulSoup(page.text, 'html')
print(soup)

<!DOCTYPE html>

<html class="client-nojs vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custom-font-size-clientpref-1 vector-feature-appearance-pinned-clientpref-1 skin-theme-clientpref-day vector-sticky-header-enabled vector-toc-not-available" dir="ltr" lang="en">
<head>
<meta charset="utf-8"/>
<title>List of postal codes of Canada: T - Wikipedia</title>
<script>(function(){var className="client-js vector-feature-language-in-header-enabled vector-feature-language-in-main-page-header-disabled vector-feature-page-tools-pinned-disabled vector-feature-toc-pinned-clientpref-1 vector-feature-main-menu-pinned-disabled vector-feature-limited-width-clientpref-1 vector-feature-limited-width-content-enabled vector-feature-custo

In [4]:
table = soup.find('table')

## Convert the table into a Data Frame

In [5]:
fsa_data = table.find_all('td')

In [6]:
def extract_fsa_detail(data):
    fsa = data.find('b').text
    
    try:
        city, *area = list(data.find('span').stripped_strings)
        area = ''.join(area)
    except ValueError:
        city = None
        area = None
    return {'fsa': fsa, 'city': city, 'area': area}

In [7]:
df = pd.DataFrame(columns = ['fsa', 'city', 'area'])

for data in fsa_data:
    df.loc[len(df)] = extract_fsa_detail(data)

df

,fsa,city,area
0,T1A,Medicine Hat,Central
1,T2A,Calgary,(Penbrooke Meadows/Marlborough)
2,T3A,Calgary,(Dalhousie/Edgemont/Hamptons/Hidden Valley)
3,T4A,Airdrie,East
4,T5A,Edmonton,(WestClareview/ EastLondonderry)
...,...,...,...
175,T5Z,Edmonton,(WestLake District)
176,T6Z,Not assigned,
177,T7Z,Stony Plain,
178,T8Z,Not assigned,


## Clean Data

In [8]:
# Drop data no city assigned
df.dropna(subset=['city'], inplace = True)
df = df[df['city'] != 'Not assigned']

In [9]:
# Add space between the slash
df.loc[:, 'area'] = df['area'].str.replace('/', ' / ')

# Remove parentheses at the start and the end
df.loc[:, 'area'] = df['area'].replace(r'^\(', '', regex=True)
df.loc[:, 'area'] = df['area'].replace(r'\)$', '', regex=True)

## Export to CSV

In [10]:
df.to_csv('data/ab_postal_codes_list.csv', index=False)